# MLOps Training 2026/2027 — Task 2
## Notebook 5 — Feature Engineering & Leakage-Proof Preprocessing Pipeline

### Prediction point
Score **after the order is placed** and payment / items / seller / customer fields plus the promised delivery date are known. Do **not** wait for payment approval, carrier pickup, customer delivery, or reviews.

`approval_delay_hours` is therefore **not** engineered: `order_approved_at` is only safe if prediction happens after approval.

### Objective
Build the feature set that will be fed directly to the model in Notebook 6:
1. **Leakage prevention** — drop every post-outcome column identified in the EDA.
2. **Feature engineering** — create new informative features from checkout-time information.
3. **Target encoding** — replace high-cardinality categoricals (zip codes, cities) with smooth
   mean late-rate estimates fitted *only* on train.
4. **OHE** — one-hot-encode low-cardinality categoricals (state, ~27 categories).
5. **Strict zero-leakage fitting** — all transformers fit on `X_train`, applied to val/test.
6. **Persist artifacts** — save fitted objects and feature matrices for Notebook 6.

### Why target encoding instead of OHE for zip/city?
- `customer_zip_code_prefix` has **13,748** unique values; OHE would produce
  13k+ binary columns that memorise the training set → massive overfitting.
- Target encoding replaces each category with its *smoothed* mean late-rate (Bayesian /
  m-estimate), giving **one** numeric column per high-cardinality feature.
- Smoothing weight `m=50` pulls rare categories toward the global mean,
  preventing memorisation of rarely-seen zip codes.

### Input Artifacts
- `artifacts/notebook_03/train.parquet`
- `artifacts/notebook_03/validation.parquet`
- `artifacts/notebook_03/test.parquet`

### Output Artifacts
- `artifacts/notebook_05/target_encoder.joblib`
- `artifacts/notebook_05/preprocessor.joblib`
- `artifacts/notebook_05/X_train.npy`, `X_validation.npy`, `X_test.npy`
- `artifacts/notebook_05/y_train.npy`, `y_validation.npy`, `y_test.npy`
- `artifacts/notebook_05/feature_config.json`, `feature_names.json`, `feature_list.csv`

## 1. Import Libraries & Configure Paths

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import save_npz, csr_matrix

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Dynamic project paths
INPUT_DIR  = Path.cwd() / "artifacts" / "notebook_03"
OUTPUT_DIR = Path.cwd() / "artifacts" / "notebook_05"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET       = "is_late"
RANDOM_STATE = 42
PREDICTION_POINT = (
    "After order placement and payment/item/seller/customer information "
    "are known, once the promised delivery date exists. Not after payment "
    "approval, carrier pickup, customer delivery, or reviews."
)

print("Input  dir:", INPUT_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Prediction point:", PREDICTION_POINT)

Input  dir: /Users/ouahibaahmid/training mlops/artifacts/notebook_03
Output dir: /Users/ouahibaahmid/training mlops/artifacts/notebook_05


## 2. Load the Train / Validation / Test Splits

In [2]:
train_df      = pd.read_parquet(INPUT_DIR / "train.parquet")
validation_df = pd.read_parquet(INPUT_DIR / "validation.parquet")
test_df       = pd.read_parquet(INPUT_DIR / "test.parquet")

print(f"Train:      {train_df.shape}")
print(f"Validation: {validation_df.shape}")
print(f"Test:       {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")

# Separate targets
y_train      = train_df[TARGET].astype(int).copy()
y_validation = validation_df[TARGET].astype(int).copy()
y_test       = test_df[TARGET].astype(int).copy()

X_train      = train_df.drop(columns=[TARGET]).copy()
X_validation = validation_df.drop(columns=[TARGET]).copy()
X_test       = test_df.drop(columns=[TARGET]).copy()

print(f"\nTrain positive rate (late): {y_train.mean():.2%}")
print(f"Val   positive rate (late): {y_validation.mean():.2%}")
print(f"Test  positive rate (late): {y_test.mean():.2%}")

Train:      (67533, 28)
Validation: (14471, 28)
Test:       (14472, 28)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'unique_products', 'unique_sellers', 'total_item_value', 'total_freight_value', 'payment_count', 'total_payment_value', 'max_payment_installments', 'review_count', 'average_review_score', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'distance_km', 'delivery_delay_days', 'is_late']

Train positive rate (late): 9.02%
Val   positive rate (late): 5.33%
Test  positive rate (late): 6.61%


## 3. Engineer Checkout-Time Features

All features below are derived **solely from information available at order placement**,
in line with the prediction-time audit from Notebook 4. `order_approved_at` is dropped
rather than turned into `approval_delay_hours`, because approval is not guaranteed at
the placement-time prediction point.

| Feature | Source | Rationale |
|---|---|---|
| `estimated_delivery_days` | estimated − purchase date | Longer promised windows → different risk profile |
| `purchase_month/weekday/hour` | purchase_timestamp | Seasonality effects found in EDA |
| `is_weekend` | purchase_timestamp weekday | Weekend orders may dispatch later |
| `freight_ratio` | freight / (item_value + 1) | Heavy/large items → higher late risk |
| `price_per_item` | item_value / item_count | Proxy for item weight/size |
| `payment_ratio` | payment_value / (item_value + 1) | Discount / installment signals |
| `is_multi_seller` | unique_sellers > 1 | Multi-seller orders harder to coordinate |
| `is_multi_product` | unique_products > 1 | Same reasoning |

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Coerce timestamps known at checkout (not order_approved_at)
    for col in ["order_purchase_timestamp", "order_estimated_delivery_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # Estimated delivery window (known at checkout)
    if "order_estimated_delivery_date" in df.columns and "order_purchase_timestamp" in df.columns:
        df["estimated_delivery_days"] = (
            df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
        ).dt.total_seconds() / (24 * 3600)

    # Purchase calendar features
    if "order_purchase_timestamp" in df.columns:
        ts = df["order_purchase_timestamp"]
        df["purchase_month"]   = ts.dt.month.astype("float32")
        df["purchase_weekday"] = ts.dt.weekday.astype("float32")
        df["purchase_hour"]    = ts.dt.hour.astype("float32")
        df["is_weekend"]       = ts.dt.weekday.isin([5, 6]).astype("float32")

    # Ratio features
    df["freight_ratio"]    = df["total_freight_value"] / (df["total_item_value"] + 1.0)
    df["price_per_item"]   = df["total_item_value"]  / (df["item_count"].replace(0, np.nan))
    df["payment_ratio"]    = df["total_payment_value"] / (df["total_item_value"] + 1.0)
    df["is_multi_seller"]  = (df["unique_sellers"]  > 1).astype("float32")
    df["is_multi_product"] = (df["unique_products"] > 1).astype("float32")

    return df


X_train      = engineer_features(X_train)
X_validation = engineer_features(X_validation)
X_test       = engineer_features(X_test)

new_cols = ["estimated_delivery_days","purchase_month",
            "purchase_weekday","purchase_hour","is_weekend",
            "freight_ratio","price_per_item","payment_ratio",
            "is_multi_seller","is_multi_product"]
print("Engineered features:", new_cols)
print(f"Total columns after engineering: {X_train.shape[1]}")

Engineered features: ['estimated_delivery_days', 'approval_delay_hours', 'purchase_month', 'purchase_weekday', 'purchase_hour', 'is_weekend', 'freight_ratio', 'price_per_item', 'payment_ratio', 'is_multi_seller', 'is_multi_product']
Total columns after engineering: 38


## 4. Drop Leakage, ID, and Raw Timestamp Columns

In [4]:
LEAKAGE_COLS = [
    # Post-outcome (used to define is_late)
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "review_count",
    "average_review_score",
    "delivery_delay_days",
    # IDs — carry no generalizable signal
    "order_id", "customer_id", "customer_unique_id",
    # Excluded per EDA audit (changes after placement)
    "order_status",
    # Not known at the placement-time prediction point
    "order_approved_at",
    # Raw timestamps — already converted to features above
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]

cols_to_drop = [c for c in LEAKAGE_COLS if c in X_train.columns]
print(f"Dropping {len(cols_to_drop)} columns: {cols_to_drop}")

X_train      = X_train.drop(columns=cols_to_drop, errors="ignore")
X_validation = X_validation.drop(columns=cols_to_drop, errors="ignore")
X_test       = X_test.drop(columns=cols_to_drop, errors="ignore")

print(f"\nRemaining {X_train.shape[1]} features:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:02d}. {col} ({X_train[col].dtype})")

Dropping 12 columns: ['order_delivered_customer_date', 'order_delivered_carrier_date', 'review_count', 'average_review_score', 'delivery_delay_days', 'order_id', 'customer_id', 'customer_unique_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_estimated_delivery_date']

Remaining 26 features:
  01. customer_zip_code_prefix (int64)
  02. customer_city (object)
  03. customer_state (object)
  04. item_count (float64)
  05. unique_products (float64)
  06. unique_sellers (float64)
  07. total_item_value (float64)
  08. total_freight_value (float64)
  09. payment_count (float64)
  10. total_payment_value (float64)
  11. max_payment_installments (float64)
  12. seller_zip_code_prefix (float64)
  13. seller_city (object)
  14. seller_state (object)
  15. distance_km (float64)
  16. estimated_delivery_days (float64)
  17. approval_delay_hours (float64)
  18. purchase_month (float32)
  19. purchase_weekday (float32)
  20. purchase_hour (float32)
  21. is_weekend (floa

## 5. Smooth Target Encoder (fit on train only)

High-cardinality columns (zip prefix > 13k unique, city > 3k unique) cannot be
OHE-encoded without exploding the feature space and memorising training set patterns.

**m-estimate formula:**
```
encoded(c) = (n_c × mean_c + m × global_mean) / (n_c + m)
```
- `n_c` = number of training samples with category `c`
- `mean_c` = late-rate for category `c` in train
- `m = 50` = smoothing weight (higher → stronger pull toward global mean for rare categories)

Unseen categories at transform time → global mean (no NaN, no leakage).

In [5]:
class SmoothTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Bayesian / m-estimate smooth target encoder.
    Fit ONLY on the training set to prevent leakage.
    """

    def __init__(self, cols, m: int = 50):
        self.cols = cols
        self.m    = m

    def fit(self, X, y):
        X_  = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        y_  = np.asarray(y, dtype=float)
        self.global_mean_ = float(y_.mean())
        self.maps_ = {}
        for col in self.cols:
            if col not in X_.columns:
                continue
            tmp = pd.DataFrame({"cat": X_[col].astype(str), "y": y_})
            stats = tmp.groupby("cat")["y"].agg(["mean", "count"])
            smoothed = (
                stats["count"] * stats["mean"] + self.m * self.global_mean_
            ) / (stats["count"] + self.m)
            self.maps_[col] = smoothed.to_dict()
        return self

    def transform(self, X):
        X_ = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        for col in self.cols:
            if col not in X_.columns:
                continue
            X_[col] = (
                X_[col].astype(str)
                       .map(self.maps_.get(col, {}))
                       .fillna(self.global_mean_)
                       .astype("float64")
            )
        return X_


# Columns that are too high-cardinality for OHE
HIGH_CARD_COLS = [c for c in ["customer_zip_code_prefix", "customer_city",
                               "seller_zip_code_prefix",  "seller_city"]
                  if c in X_train.columns]

# Low-cardinality → OHE  (customer_state ≤27, seller_state ≤20)
LOW_CARD_COLS  = [c for c in ["customer_state", "seller_state"]
                  if c in X_train.columns]

# Fit encoder on train only
te = SmoothTargetEncoder(cols=HIGH_CARD_COLS, m=50)
te.fit(X_train, y_train)

# Apply to all splits (in-place on the high-card columns)
for split in [X_train, X_validation, X_test]:
    for col in HIGH_CARD_COLS:
        if col in split.columns:
            split[col] = (
                split[col].astype(str)
                          .map(te.maps_[col])
                          .fillna(te.global_mean_)
                          .astype("float64")
            )

print(f"Target encoder fitted (m={te.m}, global mean={te.global_mean_:.4f})")
for col in HIGH_CARD_COLS:
    n_cats = len(te.maps_.get(col, {}))
    print(f"  {col}: {n_cats} categories → 1 numeric feature")

Target encoder fitted (m=50, global mean=0.0902)
  customer_zip_code_prefix: 13748 categories → 1 numeric feature
  customer_city: 3747 categories → 1 numeric feature
  seller_zip_code_prefix: 1681 categories → 1 numeric feature
  seller_city: 490 categories → 1 numeric feature


## 6. Build ColumnTransformer & Fit on Train Only

In [6]:
# After target-encoding the high-card cols are now float64 — include in numeric pipeline
ALL_NUMERIC     = [c for c in X_train.columns if c not in LOW_CARD_COLS]
ALL_CATEGORICAL = LOW_CARD_COLS

# ── Pipelines ──────────────────────────────────────────────────────────────────
# Numeric: median impute + StandardScaler
# (HistGBM doesn't need scaling, but keeping it makes the pipeline compatible
#  with any linear model as well)
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Categorical: constant impute + OHE
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot",  OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        min_frequency=0.005,   # group rare states into infrequent bucket
        max_categories=30,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric",     numeric_pipeline,     ALL_NUMERIC),
        ("categorical", categorical_pipeline, ALL_CATEGORICAL),
    ],
    remainder="drop",
)

# ── STRICT: fit ONLY on X_train ────────────────────────────────────────────────
preprocessor.fit(X_train)
print("✓ Preprocessor fitted on TRAIN only.")

X_train_t      = preprocessor.transform(X_train)
X_validation_t = preprocessor.transform(X_validation)
X_test_t       = preprocessor.transform(X_test)

feature_names_out = preprocessor.get_feature_names_out().tolist()

print(f"\nTransformed shapes:")
print(f"  Train:      {X_train_t.shape}")
print(f"  Validation: {X_validation_t.shape}")
print(f"  Test:       {X_test_t.shape}")
print(f"\nFeatures ({len(feature_names_out)}):")
for f in feature_names_out:
    print(f"  {f}")

✓ Preprocessor fitted on TRAIN only.

Transformed shapes:
  Train:      (67533, 53)
  Validation: (14471, 53)
  Test:       (14472, 53)

Features (53):
  numeric__customer_zip_code_prefix
  numeric__customer_city
  numeric__item_count
  numeric__unique_products
  numeric__unique_sellers
  numeric__total_item_value
  numeric__total_freight_value
  numeric__payment_count
  numeric__total_payment_value
  numeric__max_payment_installments
  numeric__seller_zip_code_prefix
  numeric__seller_city
  numeric__distance_km
  numeric__estimated_delivery_days
  numeric__approval_delay_hours
  numeric__purchase_month
  numeric__purchase_weekday
  numeric__purchase_hour
  numeric__is_weekend
  numeric__freight_ratio
  numeric__price_per_item
  numeric__payment_ratio
  numeric__is_multi_seller
  numeric__is_multi_product
  categorical__customer_state_BA
  categorical__customer_state_CE
  categorical__customer_state_DF
  categorical__customer_state_ES
  categorical__customer_state_GO
  categorical__cu

## 7. Persist All Artifacts

In [7]:
# ── Target encoder ──────────────────────────────────────────────────────────────
joblib.dump(te, OUTPUT_DIR / "target_encoder.joblib")
print("✓ target_encoder.joblib")

# ── Preprocessor ────────────────────────────────────────────────────────────────
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
print("✓ preprocessor.joblib")

# ── Feature matrices (dense float32 — optimal for HistGBM) ─────────────────────
np.save(OUTPUT_DIR / "X_train.npy",      X_train_t.astype("float32"))
np.save(OUTPUT_DIR / "X_validation.npy", X_validation_t.astype("float32"))
np.save(OUTPUT_DIR / "X_test.npy",       X_test_t.astype("float32"))
print("✓ X_train.npy, X_validation.npy, X_test.npy  (dense float32)")

# Also keep sparse .npz for backward compatibility
save_npz(OUTPUT_DIR / "X_train.npz",      csr_matrix(X_train_t))
save_npz(OUTPUT_DIR / "X_validation.npz", csr_matrix(X_validation_t))
save_npz(OUTPUT_DIR / "X_test.npz",       csr_matrix(X_test_t))
print("✓ X_train.npz, X_validation.npz, X_test.npz  (sparse, backward-compat)")

# ── Target arrays ────────────────────────────────────────────────────────────────
np.save(OUTPUT_DIR / "y_train.npy",      y_train.to_numpy())
np.save(OUTPUT_DIR / "y_validation.npy", y_validation.to_numpy())
np.save(OUTPUT_DIR / "y_test.npy",       y_test.to_numpy())
print("✓ y_train.npy, y_validation.npy, y_test.npy")

# ── Metadata ────────────────────────────────────────────────────────────────────
feature_config = {
    "target": TARGET,
    "prediction_point":                     PREDICTION_POINT,
    "high_cardinality_cols_target_encoded": HIGH_CARD_COLS,
    "low_cardinality_cols_ohe":             ALL_CATEGORICAL,
    "numeric_cols":                         ALL_NUMERIC,
    "leakage_cols_dropped":                 cols_to_drop,
    "excluded_post_placement_features":     ["approval_delay_hours"],
    "api_forbidden_fields": [
        "is_late",
        "delivery_delay_days",
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
        "review_count",
        "average_review_score",
        "order_approved_at",
        "approval_delay_hours",
    ],
    "target_encoder_m":                     te.m,
    "target_encoder_global_mean":           te.global_mean_,
    "num_features_out":                     len(feature_names_out),
    "random_state":                         RANDOM_STATE,
}
with open(OUTPUT_DIR / "feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)
print("✓ feature_config.json")

with open(OUTPUT_DIR / "feature_names.json", "w") as f:
    json.dump(feature_names_out, f, indent=2)
print("✓ feature_names.json")

pd.DataFrame({"feature_name": feature_names_out}).to_csv(
    OUTPUT_DIR / "feature_list.csv", index=False
)
print("✓ feature_list.csv")

✓ target_encoder.joblib
✓ preprocessor.joblib
✓ X_train.npy, X_validation.npy, X_test.npy  (dense float32)
✓ X_train.npz, X_validation.npz, X_test.npz  (sparse, backward-compat)
✓ y_train.npy, y_validation.npy, y_test.npy
✓ feature_config.json
✓ feature_names.json
✓ feature_list.csv


## 8. Verification — Reload & Check Shape

In [8]:
loaded_te  = joblib.load(OUTPUT_DIR / "target_encoder.joblib")
loaded_pre = joblib.load(OUTPUT_DIR / "preprocessor.joblib")

# Re-build validation features the same way to verify reload
val_check_df = engineer_features(validation_df.drop(columns=[TARGET])).drop(columns=cols_to_drop, errors="ignore")
for col in HIGH_CARD_COLS:
    if col in val_check_df.columns:
        val_check_df[col] = (
            val_check_df[col].astype(str)
                             .map(loaded_te.maps_[col])
                             .fillna(loaded_te.global_mean_)
                             .astype("float64")
        )
val_check_out = loaded_pre.transform(val_check_df)

assert val_check_out.shape == X_validation_t.shape, "Shape mismatch on reload!"
print(f"✓ Reload verification passed — shape: {val_check_out.shape}")

✓ Reload verification passed — shape: (14471, 53)


## 9. Summary & Key Takeaways

1. **Prediction point**: checkout / order placement — not after approval or delivery.
2. **Leakage prevention**: Post-outcome columns, reviews, `order_approved_at`, and raw timestamps dropped. `approval_delay_hours` is not created.
3. **New features**: `estimated_delivery_days`, calendar features,
   `freight_ratio`, `price_per_item`, `payment_ratio`, `is_multi_seller`, `is_multi_product`.
4. **Target encoding**: 4 high-cardinality columns → 4 numeric features (instead of 13k+ OHE).
5. **OHE**: `customer_state` and `seller_state` (low cardinality, ≤ 30 categories).
6. **Zero-leakage guarantee**: Target encoder and ColumnTransformer both fitted on `X_train` only.
7. **Artifacts**: fitted transformers, dense `.npy` + sparse `.npz`, and `feature_config.json` (includes the prediction point and API-forbidden fields).

→ Notebook 6 will load these artifacts and train/tune the final model.